In [ ]:
!pip install -q gradio transformers torch librosa soundfile SpeechRecognition

In [ ]:
import gradio as gr
import speech_recognition as sr
import librosa
import soundfile as sf
import re
import os

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis"
)

print("Sentiment model loaded!")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment model loaded!


In [ ]:
text = "I am very excited about this interview and I enjoy solving programming problems."

result = sentiment_analyzer(text)

print(result)

[{'label': 'POSITIVE', 'score': 0.9998219609260559}]


In [ ]:
def analyze_sentiment(text):
    if not text.strip():
        return "Please enter an answer."

    result = sentiment_analyzer(text)[0]

    label = result["label"]
    score = result["score"]

    return f"Sentiment: {label}\nConfidence: {score:.2%}"
print(analyze_sentiment("I am confident that I can contribute to your company."
))


Sentiment: POSITIVE
Confidence: 99.99%


In [ ]:
def basic_metrics(text):

    words = text.split()
    word_count = len(words)

    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([
        s for s in sentences if s.strip()
    ])

    return {
        "Word Count": word_count,
        "Sentence Count": sentence_count
    }
answer = """
I am a computer science student.
I enjoy programming and solving real-world problems.
I have worked on several academic projects.
"""

print(basic_metrics(answer))

{'Word Count': 20, 'Sentence Count': 3}


In [ ]:
FILLER_WORDS = [
    "um",
    "uh",
    "like",
    "actually",
    "basically",
    "you know"
]

def detect_filler_words(text):

    text_lower = text.lower()

    found = {}

    for filler in FILLER_WORDS:
        pattern = r'\b' + re.escape(filler) + r'\b'
        count = len(re.findall(pattern, text_lower))

        if count > 0:
            found[filler] = count

    total = sum(found.values())

    return found, total
answer = """
Um, I am basically a computer science student.
I, uh, like programming and actually enjoy machine learning.
"""

found, total = detect_filler_words(answer)

print("Filler words:", found)
print("Total:", total)


Filler words: {'um': 1, 'uh': 1, 'like': 1, 'actually': 1, 'basically': 1}
Total: 5


In [ ]:
STOP_WORDS = {
    "the", "is", "am", "are", "a", "an", "and",
    "to", "of", "in", "on", "for", "with", "i",
    "my", "this", "that", "it", "have", "has"
}

def extract_keywords(text, top_n=10):

    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())

    filtered = [
        word for word in words
        if word not in STOP_WORDS
    ]

    frequency = {}

    for word in filtered:
        frequency[word] = frequency.get(word, 0) + 1

    sorted_words = sorted(
        frequency.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_words[:top_n]
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I enjoy developing
machine learning projects using Python.
"""

print(extract_keywords(answer))


[('python', 2), ('machine', 2), ('learning', 2), ('computer', 1), ('science', 1), ('student', 1), ('experience', 1), ('data', 1), ('analysis', 1), ('enjoy', 1)]


In [ ]:
def calculate_score(text):

    metrics = basic_metrics(text)
    _, filler_count = detect_filler_words(text)

    word_count = metrics["Word Count"]

    score = 100

    # Very short answers
    if word_count < 20:
        score -= 20
    elif word_count < 40:
        score -= 10

    # Filler words
    score -= min(filler_count * 3, 20)

    return max(score, 0)
answer = """
I am a computer science student with experience in Python
and machine learning. I enjoy solving programming problems
and building useful applications.
"""

print("Interview Score:", calculate_score(answer))

Interview Score: 90


In [ ]:
def analyze_answer(text):

    if not text or not text.strip():
        return "Please enter an interview answer."

    sentiment = analyze_sentiment(text)
    metrics = basic_metrics(text)
    fillers, filler_count = detect_filler_words(text)
    keywords = extract_keywords(text)
    score = calculate_score(text)

    report = f"""
🎤 INTERVIEW ANALYSIS
=====================

{sentiment}

📊 Basic Metrics
Word Count: {metrics["Word Count"]}
Sentence Count: {metrics["Sentence Count"]}

⚠️ Filler Words
Total: {filler_count}
{fillers}

🔑 Keywords
{keywords}

⭐ Overall Score
{score}/100
"""

    return report
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I enjoy solving
real-world problems and developing useful applications.
"""

print(analyze_answer(answer))


🎤 INTERVIEW ANALYSIS

Sentiment: POSITIVE
Confidence: 99.90%

📊 Basic Metrics
Word Count: 24
Sentence Count: 2

⚠️ Filler Words
Total: 0
{}

🔑 Keywords
[('computer', 1), ('science', 1), ('student', 1), ('experience', 1), ('python', 1), ('machine', 1), ('learning', 1), ('data', 1), ('analysis', 1), ('enjoy', 1)]

⭐ Overall Score
90/100



In [ ]:
with gr.Blocks(title="AI Interview Analyzer") as app:

    gr.Markdown(
        """
        # 🎤 AI Interview Analyzer

        Analyze your interview answer using
        Natural Language Processing.
        """
    )

    answer_box = gr.Textbox(
        label="Enter Your Interview Answer",
        placeholder="Type your interview answer here...",
        lines=8
    )

    analyze_button = gr.Button(
        "🔍 Analyze Answer",
        variant="primary"
    )

    result_box = gr.Textbox(
        label="Analysis Result",
        lines=20
    )

    analyze_button.click(
        fn=analyze_answer,
        inputs=answer_box,
        outputs=result_box
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7150571d1956c0bd8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis"
)

print("Sentiment model loaded successfully!")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment model loaded successfully!


In [ ]:
text = "I am very excited about this interview and I enjoy solving programming problems."

result = sentiment_analyzer(text)

print(result)

[{'label': 'POSITIVE', 'score': 0.9998219609260559}]


In [ ]:
text = "I don't like programming and I am not interested in this job."

result = sentiment_analyzer(text)

print(result)

[{'label': 'NEGATIVE', 'score': 0.9996401071548462}]


In [ ]:
def analyze_sentiment(text):

    if not text.strip():
        return "Please enter an answer."

    result = sentiment_analyzer(text)[0]

    label = result["label"]
    score = result["score"]

    return f"Sentiment: {label}\nConfidence: {score:.2%}"

In [ ]:
answer = "I am confident that I can contribute to your company."

print(analyze_sentiment(answer))

Sentiment: POSITIVE
Confidence: 99.99%


In [ ]:
import re

def basic_metrics(text):

    words = text.split()
    word_count = len(words)

    sentences = re.split(r'[.!?]+', text)

    sentence_count = len([
        sentence
        for sentence in sentences
        if sentence.strip()
    ])

    return {
        "Word Count": word_count,
        "Sentence Count": sentence_count
    }

In [ ]:
answer = """
I am a computer science student.
I enjoy programming and solving real-world problems.
I have worked on several academic projects.
"""

metrics = basic_metrics(answer)

print(metrics)


{'Word Count': 20, 'Sentence Count': 3}


In [ ]:
FILLER_WORDS = [
    "um",
    "uh",
    "like",
    "actually",
    "basically",
    "you know"
]

def detect_filler_words(text):

    text_lower = text.lower()

    found = {}

    for filler in FILLER_WORDS:

        pattern = r'\b' + re.escape(filler) + r'\b'

        count = len(
            re.findall(pattern, text_lower)
        )

        if count > 0:
            found[filler] = count

    total = sum(found.values())

    return found, total
answer = """
Um, I am basically a computer science student.
I, uh, like programming and actually enjoy machine learning.
"""

found, total = detect_filler_words(answer)

print("Filler words:", found)
print("Total filler words:", total)


Filler words: {'um': 1, 'uh': 1, 'like': 1, 'actually': 1, 'basically': 1}
Total filler words: 5


In [ ]:
STOP_WORDS = {
    "the", "is", "am", "are", "a", "an",
    "and", "to", "of", "in", "on", "for",
    "with", "i", "my", "this", "that",
    "it", "have", "has"
}

def extract_keywords(text, top_n=10):

    words = re.findall(
        r'\b[a-zA-Z]{3,}\b',
        text.lower()
    )

    filtered = [
        word
        for word in words
        if word not in STOP_WORDS
    ]

    frequency = {}

    for word in filtered:
        frequency[word] = frequency.get(word, 0) + 1

    sorted_words = sorted(
        frequency.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_words[:top_n]
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I enjoy developing
machine learning projects using Python.
"""

keywords = extract_keywords(answer)

print(keywords)


[('python', 2), ('machine', 2), ('learning', 2), ('computer', 1), ('science', 1), ('student', 1), ('experience', 1), ('data', 1), ('analysis', 1), ('enjoy', 1)]


In [ ]:
def calculate_score(text):

    metrics = basic_metrics(text)
    _, filler_count = detect_filler_words(text)

    word_count = metrics["Word Count"]

    # Start with 100 points
    score = 100

    # Check answer length
    if word_count < 20:
        score -= 20
    elif word_count < 40:
        score -= 10

    # Penalize filler words
    score -= min(filler_count * 3, 20)

    # Keep score between 0 and 100
    score = max(0, min(score, 100))

    return score
answer = """
I am a computer science student with experience in Python
and machine learning. I enjoy solving programming problems
and building useful applications.
"""

score = calculate_score(answer)

print("Interview Score:", score, "/100")


Interview Score: 90 /100


In [ ]:
def generate_recommendations(text):

    recommendations = []

    metrics = basic_metrics(text)
    _, filler_count = detect_filler_words(text)

    word_count = metrics["Word Count"]

    if word_count < 20:
        recommendations.append(
            "Give a more detailed answer."
        )

    elif word_count < 40:
        recommendations.append(
            "Consider adding a specific example."
        )

    if filler_count >= 3:
        recommendations.append(
            "Try to reduce filler words such as 'um', 'uh' and 'like'."
        )

    if filler_count == 0:
        recommendations.append(
            "Good job avoiding common filler words."
        )

    if word_count >= 40:
        recommendations.append(
            "Your answer has good detail. Keep it focused and relevant."
        )

    if not recommendations:
        recommendations.append(
            "Good answer. Continue practicing for better confidence."
        )

    return recommendations
answer = """
Um, I am basically a computer science student and
I like programming. I am actually interested in
machine learning.
"""

recommendations = generate_recommendations(answer)

for recommendation in recommendations:
    print("•", recommendation)


• Give a more detailed answer.
• Try to reduce filler words such as 'um', 'uh' and 'like'.


In [ ]:
def analyze_answer(text):

    if not text or not text.strip():
        return "Please enter an interview answer."

    # Sentiment
    sentiment = analyze_sentiment(text)

    # Basic metrics
    metrics = basic_metrics(text)

    # Filler words
    fillers, filler_count = detect_filler_words(text)

    # Keywords
    keywords = extract_keywords(text)

    # Score
    score = calculate_score(text)

    # Recommendations
    recommendations = generate_recommendations(text)

    # Format recommendations
    recommendation_text = "\n".join(
        "• " + item
        for item in recommendations
    )

    report = f"""
🎤 INTERVIEW ANALYSIS
==============================

😊 SENTIMENT
{sentiment}

📊 BASIC METRICS
Word Count: {metrics["Word Count"]}
Sentence Count: {metrics["Sentence Count"]}

⚠️ FILLER WORDS
Total: {filler_count}
{fillers}

🔑 KEYWORDS
{keywords}

⭐ OVERALL SCORE
{score}/100

💡 RECOMMENDATIONS
{recommendation_text}
"""

    return report
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I have worked on several
academic projects and I enjoy solving real-world problems.
I am interested in developing useful software applications
and improving my programming skills.
"""

result = analyze_answer(answer)

print(result)



🎤 INTERVIEW ANALYSIS

😊 SENTIMENT
Sentiment: POSITIVE
Confidence: 99.93%

📊 BASIC METRICS
Word Count: 41
Sentence Count: 3

⚠️ FILLER WORDS
Total: 0
{}

🔑 KEYWORDS
[('computer', 1), ('science', 1), ('student', 1), ('experience', 1), ('python', 1), ('machine', 1), ('learning', 1), ('data', 1), ('analysis', 1), ('worked', 1)]

⭐ OVERALL SCORE
100/100

💡 RECOMMENDATIONS
• Good job avoiding common filler words.
• Your answer has good detail. Keep it focused and relevant.



In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks(
    title="AI Interview Analyzer"
) as app:

    gr.Markdown(
        """
        # 🎤 AI Interview Analyzer

        ### Analyze your interview answer using AI

        Enter your answer below and click **Analyze Answer**.
        """
    )

    answer_box = gr.Textbox(
        label="📝 Interview Answer",
        placeholder="Type your interview answer here...",
        lines=10
    )

    analyze_button = gr.Button(
        "🔍 Analyze Answer",
        variant="primary"
    )

    result_box = gr.Textbox(
        label="📊 Analysis Result",
        lines=25
    )

    analyze_button.click(
        fn=analyze_answer,
        inputs=answer_box,
        outputs=result_box
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84fa707697c879e4dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def calculate_improved_score(text):

    if not text or not text.strip():
        return 0

    # -------------------------
    # 1. Answer Length - 20
    # -------------------------
    word_count = basic_metrics(text)["Word Count"]

    if word_count >= 60:
        length_score = 20
    elif word_count >= 40:
        length_score = 15
    elif word_count >= 20:
        length_score = 10
    else:
        length_score = 5

    # -------------------------
    # 2. Sentiment - 20
    # -------------------------
    sentiment_result = sentiment_analyzer(text)[0]

    if sentiment_result["label"] == "POSITIVE":
        sentiment_score = 20
    else:
        sentiment_score = 10

    # -------------------------
    # 3. Keywords - 20
    # -------------------------
    keywords = extract_keywords(text)

    if len(keywords) >= 8:
        keyword_score = 20
    elif len(keywords) >= 5:
        keyword_score = 15
    elif len(keywords) >= 3:
        keyword_score = 10
    else:
        keyword_score = 5

    # -------------------------
    # 4. Filler Words - 20
    # -------------------------
    _, filler_count = detect_filler_words(text)

    if filler_count == 0:
        filler_score = 20
    elif filler_count <= 2:
        filler_score = 15
    elif filler_count <= 4:
        filler_score = 10
    else:
        filler_score = 5

    # -------------------------
    # 5. Structure - 20
    # -------------------------
    sentence_count = basic_metrics(text)["Sentence Count"]

    if sentence_count >= 5:
        structure_score = 20
    elif sentence_count >= 3:
        structure_score = 15
    elif sentence_count >= 2:
        structure_score = 10
    else:
        structure_score = 5

    # -------------------------
    # Final Score
    # -------------------------
    total_score = (
        length_score +
        sentiment_score +
        keyword_score +
        filler_score +
        structure_score
    )

    return total_score
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I have worked on several
academic projects where I developed applications to solve
real-world problems. I enjoy learning new technologies,
working with teams, and improving my programming skills.
"""

score = calculate_improved_score(answer)

print("Improved Interview Score:", score, "/100")


Improved Interview Score: 90 /100


In [ ]:
def get_performance_level(score):

    if score >= 90:
        return "Excellent 🌟"

    elif score >= 75:
        return "Very Good 👍"

    elif score >= 60:
        return "Good 🙂"

    elif score >= 40:
        return "Needs Improvement ⚠️"

    else:
        return "Poor ❗"
score = 82

print(get_performance_level(score))


Very Good 👍


In [ ]:
def analyze_answer(text):

    if not text or not text.strip():
        return "Please enter an interview answer."

    # Sentiment
    sentiment = analyze_sentiment(text)

    # Metrics
    metrics = basic_metrics(text)

    # Filler words
    fillers, filler_count = detect_filler_words(text)

    # Keywords
    keywords = extract_keywords(text)

    # Improved score
    score = calculate_improved_score(text)

    # Performance level
    performance = get_performance_level(score)

    # Recommendations
    recommendations = generate_recommendations(text)

    recommendation_text = "\n".join(
        "• " + item
        for item in recommendations
    )

    keyword_text = ", ".join(
        word for word, count in keywords
    )

    filler_text = ", ".join(
        f"{word}: {count}"
        for word, count in fillers.items()
    )

    if not filler_text:
        filler_text = "None"

    report = f"""
🎤 AI INTERVIEW ANALYSIS
========================================

⭐ OVERALL SCORE
{score}/100

🏆 PERFORMANCE LEVEL
{performance}


😊 SENTIMENT
{sentiment}


📊 ANSWER METRICS
Word Count: {metrics["Word Count"]}
Sentence Count: {metrics["Sentence Count"]}


⚠️ FILLER WORDS
{filler_text}
Total: {filler_count}


🔑 IMPORTANT KEYWORDS
{keyword_text}


💡 RECOMMENDATIONS
{recommendation_text}
"""

    return report
answer = """
I am a computer science student with strong interest in
Python and machine learning. I have completed several
academic projects involving data analysis and application
development. I enjoy solving real-world problems and
learning new technologies. I also enjoy working with
team members and contributing ideas to projects.
"""

print(analyze_answer(answer))



🎤 AI INTERVIEW ANALYSIS

⭐ OVERALL SCORE
90/100

🏆 PERFORMANCE LEVEL
Excellent 🌟


😊 SENTIMENT
Sentiment: POSITIVE
Confidence: 99.96%


📊 ANSWER METRICS
Word Count: 47
Sentence Count: 4


⚠️ FILLER WORDS
None
Total: 0


🔑 IMPORTANT KEYWORDS
learning, projects, enjoy, computer, science, student, strong, interest, python, machine


💡 RECOMMENDATIONS
• Good job avoiding common filler words.
• Your answer has good detail. Keep it focused and relevant.



In [ ]:
def get_analysis_data(text):

    if not text or not text.strip():
        return None

    # Basic analysis
    sentiment_result = sentiment_analyzer(text)[0]

    metrics = basic_metrics(text)

    fillers, filler_count = detect_filler_words(text)

    keywords = extract_keywords(text)

    score = calculate_improved_score(text)

    performance = get_performance_level(score)

    recommendations = generate_recommendations(text)

    return {
        "sentiment": sentiment_result["label"],
        "sentiment_confidence": sentiment_result["score"],
        "word_count": metrics["Word Count"],
        "sentence_count": metrics["Sentence Count"],
        "fillers": fillers,
        "filler_count": filler_count,
        "keywords": keywords,
        "score": score,
        "performance": performance,
        "recommendations": recommendations
    }
answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I have worked on several
academic projects and I enjoy solving real-world problems.
I am interested in developing useful software applications.
"""

data = get_analysis_data(answer)

print(data)


{'sentiment': 'POSITIVE', 'sentiment_confidence': 0.9991580247879028, 'word_count': 36, 'sentence_count': 3, 'fillers': {}, 'filler_count': 0, 'keywords': [('computer', 1), ('science', 1), ('student', 1), ('experience', 1), ('python', 1), ('machine', 1), ('learning', 1), ('data', 1), ('analysis', 1), ('worked', 1)], 'score': 85, 'performance': 'Very Good 👍', 'recommendations': ['Consider adding a specific example.', 'Good job avoiding common filler words.']}


In [ ]:
!pip install -q plotly

In [ ]:
import plotly.graph_objects as go

print("Plotly loaded successfully!")

Plotly loaded successfully!


In [ ]:
def create_score_chart(score):

    remaining = 100 - score

    fig = go.Figure(
        go.Pie(
            values=[score, remaining],
            labels=["Score", "Remaining"],
            hole=0.65,
            marker=dict(
                colors=["#4CAF50", "#E0E0E0"]
            )
        )
    )

    fig.update_layout(
        title="Interview Performance Score",
        showlegend=True
    )

    return fig
score_chart = create_score_chart(82)

score_chart.show()


In [ ]:
def create_filler_chart(fillers):

    if not fillers:
        fig = go.Figure()

        fig.add_annotation(
            text="No filler words detected 🎉",
            x=0.5,
            y=0.5,
            showarrow=False,
            font=dict(size=20)
        )

        fig.update_xaxes(visible=False)
        fig.update_yaxes(visible=False)

        return fig

    words = list(fillers.keys())
    counts = list(fillers.values())

    fig = go.Figure(
        go.Bar(
            x=words,
            y=counts,
            marker_color="#FF9800"
        )
    )

    fig.update_layout(
        title="Filler Words Detected",
        xaxis_title="Filler Word",
        yaxis_title="Count"
    )

    return fig
test_fillers = {
    "um": 3,
    "like": 2,
    "actually": 1
}

filler_chart = create_filler_chart(test_fillers)

filler_chart.show()


In [ ]:
def create_metrics_chart(word_count, sentence_count, filler_count):

    fig = go.Figure(
        go.Bar(
            x=["Words", "Sentences", "Filler Words"],
            y=[word_count, sentence_count, filler_count],
            marker_color=[
                "#2196F3",
                "#9C27B0",
                "#FF5722"
            ]
        )
    )

    fig.update_layout(
        title="Answer Metrics",
        yaxis_title="Count"
    )

    return fig
metrics_chart = create_metrics_chart(
    75,
    6,
    2
)

metrics_chart.show()


In [ ]:
import gradio as gr


In [ ]:
def run_full_analysis(text):

    if not text or not text.strip():

        return (
            "Please enter an interview answer.",
            0,
            "N/A",
            None,
            None,
            None
        )

    data = get_analysis_data(text)

    # Performance text
    performance_text = (
        f"{data['performance']}\n"
        f"Sentiment: {data['sentiment']}\n"
        f"Confidence: {data['sentiment_confidence']:.2%}\n"
        f"Words: {data['word_count']}\n"
        f"Sentences: {data['sentence_count']}\n"
        f"Filler Words: {data['filler_count']}"
    )

    # Recommendations
    recommendations = "\n".join(
        "• " + r
        for r in data["recommendations"]
    )

    # Keywords
    keyword_text = ", ".join(
        word for word, count in data["keywords"]
    )

    # Score chart
    score_chart = create_score_chart(
        data["score"]
    )

    # Filler chart
    filler_chart = create_filler_chart(
        data["fillers"]
    )

    # Metrics chart
    metrics_chart = create_metrics_chart(
        data["word_count"],
        data["sentence_count"],
        data["filler_count"]
    )

    report = f"""
## 🎤 Interview Analysis

### ⭐ Performance

{performance_text}

### 🔑 Important Keywords

{keyword_text}

### 💡 Recommendations

{recommendations}
"""

    return (
        report,
        data["score"],
        data["performance"],
        score_chart,
        filler_chart,
        metrics_chart
    )


In [ ]:
with gr.Blocks(
    title="AI Interview Performance Analyzer",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # 🎤 AI Interview Performance Analyzer

        ### Analyze your interview answer using Natural Language Processing

        Enter your answer below and receive an AI-powered performance analysis.
        """
    )

    with gr.Row():

        with gr.Column(scale=2):

            question = gr.Textbox(
                label="📝 Interview Question",
                placeholder="Example: Tell me about yourself.",
                lines=2
            )

            answer = gr.Textbox(
                label="✍️ Your Interview Answer",
                placeholder="Type your answer here...",
                lines=12
            )

            analyze_button = gr.Button(
                "🔍 Analyze Answer",
                variant="primary"
            )

        with gr.Column(scale=1):

            score = gr.Number(
                label="⭐ Overall Score",
                value=0,
                precision=0
            )

            performance = gr.Textbox(
                label="🏆 Performance"
            )

    report = gr.Markdown(
        label="📄 Analysis Report"
    )

    gr.Markdown("## 📊 Visual Analysis")

    with gr.Row():

        score_plot = gr.Plot(
            label="⭐ Score"
        )

        filler_plot = gr.Plot(
            label="⚠️ Filler Words"
        )

        metrics_plot = gr.Plot(
            label="📊 Metrics"
        )

    analyze_button.click(
        fn=run_full_analysis,
        inputs=answer,
        outputs=[
            report,
            score,
            performance,
            score_plot,
            filler_plot,
            metrics_plot
        ]
    )

app.launch(share=True)


/tmp/ipykernel_4688/3892814751.py:1: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c03b9d4c2bc717b64.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer, util

similarity_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Similarity model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity model loaded successfully!


In [ ]:
def calculate_relevance(question, answer):

    if not question.strip() or not answer.strip():
        return 0

    question_embedding = similarity_model.encode(
        question,
        convert_to_tensor=True
    )

    answer_embedding = similarity_model.encode(
        answer,
        convert_to_tensor=True
    )

    similarity = util.cos_sim(
        question_embedding,
        answer_embedding
    )

    score = float(similarity[0][0])

    # Convert similarity to 0-100
    relevance_score = max(
        0,
        min(100, score * 100)
    )

    return round(relevance_score, 2)
question = "Why should we hire you?"

answer = """
I have strong programming skills and experience in Python
and machine learning. I have completed several projects
and I believe I can contribute to your development team.
"""

score = calculate_relevance(
    question,
    answer
)

print("Relevance Score:", score, "/100")
question = "Why should we hire you?"

answer = """
I have strong programming skills and experience in Python
and machine learning. I have completed several projects
and I believe I can contribute to your development team.
"""

score = calculate_relevance(
    question,
    answer
)

print("Relevance Score:", score, "/100")



Relevance Score: 32.8 /100
Relevance Score: 32.8 /100


In [ ]:
def calculate_final_score(question, text):

    if not text or not text.strip():
        return 0

    # -------------------------
    # 1. Answer Length - 15
    # -------------------------
    word_count = basic_metrics(text)["Word Count"]

    if word_count >= 60:
        length_score = 15
    elif word_count >= 40:
        length_score = 12
    elif word_count >= 20:
        length_score = 8
    else:
        length_score = 4

    # -------------------------
    # 2. Sentiment - 15
    # -------------------------
    sentiment_result = sentiment_analyzer(text)[0]

    if sentiment_result["label"] == "POSITIVE":
        sentiment_score = 15
    else:
        sentiment_score = 8

    # -------------------------
    # 3. Keywords - 15
    # -------------------------
    keywords = extract_keywords(text)

    if len(keywords) >= 8:
        keyword_score = 15
    elif len(keywords) >= 5:
        keyword_score = 12
    elif len(keywords) >= 3:
        keyword_score = 8
    else:
        keyword_score = 4

    # -------------------------
    # 4. Filler Words - 15
    # -------------------------
    _, filler_count = detect_filler_words(text)

    if filler_count == 0:
        filler_score = 15
    elif filler_count <= 2:
        filler_score = 12
    elif filler_count <= 4:
        filler_score = 8
    else:
        filler_score = 4

    # -------------------------
    # 5. Structure - 15
    # -------------------------
    sentence_count = basic_metrics(text)["Sentence Count"]

    if sentence_count >= 5:
        structure_score = 15
    elif sentence_count >= 3:
        structure_score = 12
    elif sentence_count >= 2:
        structure_score = 8
    else:
        structure_score = 4

    # -------------------------
    # 6. Relevance - 25
    # -------------------------
    relevance = calculate_relevance(
        question,
        text
    )

    relevance_score = relevance * 0.25

    # -------------------------
    # Final Score
    # -------------------------
    total_score = (
        length_score +
        sentiment_score +
        keyword_score +
        filler_score +
        structure_score +
        relevance_score
    )

    return round(
        max(0, min(100, total_score)),
        2
    )
question = "Why should we hire you?"

answer = """
I have strong programming skills and experience with Python
and machine learning. I have completed several academic and
personal projects where I solved real-world problems.
I am a quick learner and I believe I can contribute
effectively to your development team.
"""

score = calculate_final_score(
    question,
    answer
)

print("Final Interview Score:", score, "/100")


Final Interview Score: 77.32 /100


In [ ]:
def get_analysis_data(question, text):

    if not text or not text.strip():
        return None

    # Sentiment
    sentiment_result = sentiment_analyzer(text)[0]

    # Basic metrics
    metrics = basic_metrics(text)

    # Filler words
    fillers, filler_count = detect_filler_words(text)

    # Keywords
    keywords = extract_keywords(text)

    # Relevance
    relevance = calculate_relevance(
        question,
        text
    )

    # Final score
    score = calculate_final_score(
        question,
        text
    )

    # Performance
    performance = get_performance_level(score)

    # Recommendations
    recommendations = generate_recommendations(text)

    # Add relevance recommendation
    if relevance < 50:
        recommendations.append(
            "Your answer may not directly address the interview question."
        )
    elif relevance >= 75:
        recommendations.append(
            "Your answer is strongly related to the interview question."
        )

    return {
        "sentiment": sentiment_result["label"],
        "sentiment_confidence": sentiment_result["score"],
        "word_count": metrics["Word Count"],
        "sentence_count": metrics["Sentence Count"],
        "fillers": fillers,
        "filler_count": filler_count,
        "keywords": keywords,
        "relevance": relevance,
        "score": score,
        "performance": performance,
        "recommendations": recommendations
    }


In [ ]:
def run_full_analysis(question, text):

    if not question or not question.strip():
        return (
            "Please enter an interview question.",
            0,
            "N/A",
            None,
            None,
            None
        )

    if not text or not text.strip():
        return (
            "Please enter your interview answer.",
            0,
            "N/A",
            None,
            None,
            None
        )

    data = get_analysis_data(
        question,
        text
    )

    performance_text = (
        f"{data['performance']}\n"
        f"Sentiment: {data['sentiment']}\n"
        f"Sentiment Confidence: "
        f"{data['sentiment_confidence']:.2%}\n"
        f"Question Relevance: "
        f"{data['relevance']:.2f}/100\n"
        f"Words: {data['word_count']}\n"
        f"Sentences: {data['sentence_count']}\n"
        f"Filler Words: {data['filler_count']}"
    )

    recommendations = "\n".join(
        "• " + r
        for r in data["recommendations"]
    )

    keyword_text = ", ".join(
        word
        for word, count in data["keywords"]
    )

    score_chart = create_score_chart(
        data["score"]
    )

    filler_chart = create_filler_chart(
        data["fillers"]
    )

    metrics_chart = create_metrics_chart(
        data["word_count"],
        data["sentence_count"],
        data["filler_count"]
    )

    report = f"""
## 🎤 Interview Analysis

### ⭐ Performance

{performance_text}

### 🔑 Important Keywords

{keyword_text}

### 💡 Recommendations

{recommendations}
"""

    return (
        report,
        data["score"],
        data["performance"],
        score_chart,
        filler_chart,
        metrics_chart
    )


In [ ]:
with gr.Blocks(
    title="AI Interview Performance Analyzer",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # 🎤 AI Interview Performance Analyzer

        ### NLP-based analysis of interview answers
        """
    )

    with gr.Row():

        with gr.Column(scale=2):

            question = gr.Textbox(
                label="📝 Interview Question",
                placeholder="Example: Why should we hire you?",
                lines=3
            )

            answer = gr.Textbox(
                label="✍️ Your Interview Answer",
                placeholder="Type your answer here...",
                lines=12
            )

            analyze_button = gr.Button(
                "🔍 Analyze Answer",
                variant="primary"
            )

        with gr.Column(scale=1):

            score = gr.Number(
                label="⭐ Overall Score",
                value=0,
                precision=2
            )

            performance = gr.Textbox(
                label="🏆 Performance Level"
            )

    report = gr.Markdown(
        label="📄 Analysis Report"
    )

    gr.Markdown(
        "## 📊 Visual Analysis"
    )

    with gr.Row():

        score_plot = gr.Plot(
            label="⭐ Score"
        )

        filler_plot = gr.Plot(
            label="⚠️ Filler Words"
        )

        metrics_plot = gr.Plot(
            label="📊 Answer Metrics"
        )

    analyze_button.click(
        fn=run_full_analysis,
        inputs=[
            question,
            answer
        ],
        outputs=[
            report,
            score,
            performance,
            score_plot,
            filler_plot,
            metrics_plot
        ]
    )

app.launch(share=True)


/tmp/ipykernel_4688/3968890770.py:1: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5990bd0f7d5aab976.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def generate_report(question, text):

    if not question.strip():
        return None

    if not text.strip():
        return None

    data = get_analysis_data(
        question,
        text
    )

    keyword_text = ", ".join(
        word
        for word, count in data["keywords"]
    )

    filler_text = ", ".join(
        f"{word}: {count}"
        for word, count in data["fillers"].items()
    )

    if not filler_text:
        filler_text = "None"

    recommendation_text = "\n".join(
        f"- {item}"
        for item in data["recommendations"]
    )

    report = f"""
==================================================
        AI INTERVIEW PERFORMANCE REPORT
==================================================

INTERVIEW QUESTION
--------------------------------------------------
{question}

CANDIDATE ANSWER
--------------------------------------------------
{text}

OVERALL PERFORMANCE
--------------------------------------------------
Score              : {data["score"]}/100
Performance Level  : {data["performance"]}

SENTIMENT ANALYSIS
--------------------------------------------------
Sentiment           : {data["sentiment"]}
Confidence          : {data["sentiment_confidence"]:.2%}

QUESTION RELEVANCE
--------------------------------------------------
Relevance Score     : {data["relevance"]:.2f}/100

ANSWER METRICS
--------------------------------------------------
Word Count          : {data["word_count"]}
Sentence Count      : {data["sentence_count"]}

FILLER WORD ANALYSIS
--------------------------------------------------
Total Filler Words  : {data["filler_count"]}
Detected Words      : {filler_text}

KEYWORDS
--------------------------------------------------
{keyword_text}

RECOMMENDATIONS
--------------------------------------------------
{recommendation_text}

==================================================
             END OF REPORT
==================================================
"""

    filename = "interview_analysis_report.txt"

    with open(filename, "w", encoding="utf-8") as file:
        file.write(report)

    return filename
question = "Why should we hire you?"

answer = """
I am a computer science student with experience in Python,
machine learning and data analysis. I have completed several
projects and enjoy solving real-world problems. I am a quick
learner and believe I can contribute to your development team.
"""

report_file = generate_report(
    question,
    answer
)

print("Report created:", report_file)


Report created: interview_analysis_report.txt


In [ ]:
with open(
    "interview_analysis_report.txt",
    "r",
    encoding="utf-8"
) as file:

    print(file.read())



        AI INTERVIEW PERFORMANCE REPORT

INTERVIEW QUESTION
--------------------------------------------------
Why should we hire you?

CANDIDATE ANSWER
--------------------------------------------------

I am a computer science student with experience in Python,
machine learning and data analysis. I have completed several
projects and enjoy solving real-world problems. I am a quick
learner and believe I can contribute to your development team.


OVERALL PERFORMANCE
--------------------------------------------------
Score              : 73.11/100
Performance Level  : Good 🙂

SENTIMENT ANALYSIS
--------------------------------------------------
Sentiment           : POSITIVE
Confidence          : 99.97%

QUESTION RELEVANCE
--------------------------------------------------
Relevance Score     : 32.42/100

ANSWER METRICS
--------------------------------------------------
Word Count          : 39
Sentence Count      : 3

FILLER WORD ANALYSIS
----------------------------------------------

In [ ]:
def run_complete_analysis(question, text):

    if not question or not question.strip():

        return (
            "Please enter an interview question.",
            0,
            "N/A",
            None,
            None,
            None,
            None
        )

    if not text or not text.strip():

        return (
            "Please enter your interview answer.",
            0,
            "N/A",
            None,
            None,
            None,
            None
        )

    # Get analysis
    data = get_analysis_data(
        question,
        text
    )

    # Performance information
    performance_text = (
        f"{data['performance']}\n"
        f"Sentiment: {data['sentiment']}\n"
        f"Confidence: "
        f"{data['sentiment_confidence']:.2%}\n"
        f"Question Relevance: "
        f"{data['relevance']:.2f}/100\n"
        f"Words: {data['word_count']}\n"
        f"Sentences: {data['sentence_count']}\n"
        f"Filler Words: {data['filler_count']}"
    )

    # Recommendations
    recommendations = "\n".join(
        "• " + r
        for r in data["recommendations"]
    )

    # Keywords
    keyword_text = ", ".join(
        word
        for word, count in data["keywords"]
    )

    # Report
    report = f"""
## 🎤 Interview Analysis

### ⭐ Performance

{performance_text}

### 🔑 Important Keywords

{keyword_text}

### 💡 Recommendations

{recommendations}
"""

    # Charts
    score_chart = create_score_chart(
        data["score"]
    )

    filler_chart = create_filler_chart(
        data["fillers"]
    )

    metrics_chart = create_metrics_chart(
        data["word_count"],
        data["sentence_count"],
        data["filler_count"]
    )

    # Generate downloadable report
    report_file = generate_report(
        question,
        text
    )

    return (
        report,
        data["score"],
        data["performance"],
        score_chart,
        filler_chart,
        metrics_chart,
        report_file
    )


In [ ]:
with gr.Blocks(
    title="AI Interview Performance Analyzer",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # 🎤 AI Interview Performance Analyzer

        ### Evaluate your interview answer using Natural Language Processing

        Enter an interview question and your answer below.
        The system analyzes sentiment, relevance, keywords,
        filler words and overall performance.
        """
    )

    # -----------------------------
    # INPUT SECTION
    # -----------------------------

    gr.Markdown("## 📝 Interview Input")

    with gr.Row():

        with gr.Column():

            question = gr.Textbox(
                label="Interview Question",
                placeholder="Example: Why should we hire you?",
                lines=3
            )

        with gr.Column():

            answer = gr.Textbox(
                label="Your Interview Answer",
                placeholder="Type your answer here...",
                lines=8
            )

    analyze_button = gr.Button(
        "🔍 ANALYZE ANSWER",
        variant="primary",
        size="lg"
    )

    # -----------------------------
    # SCORE SECTION
    # -----------------------------

    gr.Markdown("## ⭐ Performance Summary")

    with gr.Row():

        score = gr.Number(
            label="Overall Score",
            value=0,
            precision=2
        )

        performance = gr.Textbox(
            label="Performance Level"
        )

    # -----------------------------
    # REPORT
    # -----------------------------

    gr.Markdown("## 📄 Analysis Report")

    report = gr.Markdown()

    # -----------------------------
    # CHARTS
    # -----------------------------

    gr.Markdown("## 📊 Visual Analysis")

    with gr.Row():

        score_plot = gr.Plot(
            label="Performance Score"
        )

        filler_plot = gr.Plot(
            label="Filler Words"
        )

        metrics_plot = gr.Plot(
            label="Answer Metrics"
        )

    # -----------------------------
    # DOWNLOAD
    # -----------------------------

    gr.Markdown("## 📥 Download Report")

    report_file = gr.File(
        label="Download Interview Report"
    )

    # -----------------------------
    # BUTTON ACTION
    # -----------------------------

    analyze_button.click(
        fn=run_complete_analysis,

        inputs=[
            question,
            answer
        ],

        outputs=[
            report,
            score,
            performance,
            score_plot,
            filler_plot,
            metrics_plot,
            report_file
        ]
    )

app.launch(share=True)


/tmp/ipykernel_4688/1211701135.py:1: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36e9f65854ec779878.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def get_relevance_level(score):

    if score >= 80:
        return "Highly Relevant 🎯"

    elif score >= 60:
        return "Relevant 👍"

    elif score >= 40:
        return "Partially Relevant ⚠️"

    else:
        return "Low Relevance ❌"
print(get_relevance_level(85))
print(get_relevance_level(65))
print(get_relevance_level(45))
print(get_relevance_level(25))


Highly Relevant 🎯
Relevant 👍
Partially Relevant ⚠️
Low Relevance ❌


In [ ]:
def get_analysis_data(question, text):

    if not text or not text.strip():
        return None

    sentiment_result = sentiment_analyzer(text)[0]

    metrics = basic_metrics(text)

    fillers, filler_count = detect_filler_words(text)

    keywords = extract_keywords(text)

    relevance = calculate_relevance(
        question,
        text
    )

    relevance_level = get_relevance_level(
        relevance
    )

    score = calculate_final_score(
        question,
        text
    )

    performance = get_performance_level(
        score
    )

    recommendations = generate_recommendations(
        text
    )

    if relevance < 50:
        recommendations.append(
            "Make sure your answer directly addresses the interview question."
        )

    elif relevance >= 80:
        recommendations.append(
            "Excellent! Your answer is strongly connected to the question."
        )

    return {
        "sentiment": sentiment_result["label"],
        "sentiment_confidence": sentiment_result["score"],
        "word_count": metrics["Word Count"],
        "sentence_count": metrics["Sentence Count"],
        "fillers": fillers,
        "filler_count": filler_count,
        "keywords": keywords,
        "relevance": relevance,
        "relevance_level": relevance_level,
        "score": score,
        "performance": performance,
        "recommendations": recommendations
    }


In [ ]:
def generate_report(question, text):

    if not question.strip():
        return None

    if not text.strip():
        return None

    data = get_analysis_data(
        question,
        text
    )

    keyword_text = ", ".join(
        word
        for word, count in data["keywords"]
    )

    filler_text = ", ".join(
        f"{word}: {count}"
        for word, count in data["fillers"].items()
    )

    if not filler_text:
        filler_text = "None"

    recommendation_text = "\n".join(
        f"- {item}"
        for item in data["recommendations"]
    )

    report = f"""
==================================================
       AI INTERVIEW PERFORMANCE REPORT
==================================================

INTERVIEW QUESTION
--------------------------------------------------
{question}

CANDIDATE ANSWER
--------------------------------------------------
{text}

OVERALL PERFORMANCE
--------------------------------------------------
Score              : {data["score"]}/100
Performance Level  : {data["performance"]}

SENTIMENT ANALYSIS
--------------------------------------------------
Sentiment           : {data["sentiment"]}
Confidence          : {data["sentiment_confidence"]:.2%}

QUESTION RELEVANCE
--------------------------------------------------
Relevance Score     : {data["relevance"]:.2f}/100
Relevance Level     : {data["relevance_level"]}

ANSWER METRICS
--------------------------------------------------
Word Count          : {data["word_count"]}
Sentence Count      : {data["sentence_count"]}

FILLER WORD ANALYSIS
--------------------------------------------------
Total Filler Words  : {data["filler_count"]}
Detected Words      : {filler_text}

KEYWORDS
--------------------------------------------------
{keyword_text}

RECOMMENDATIONS
--------------------------------------------------
{recommendation_text}

==================================================
                 END OF REPORT
==================================================
"""

    filename = "interview_analysis_report.txt"

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(report)

    return filename


In [ ]:
app.launch(share=True)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36e9f65854ec779878.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
